In [1]:

class Problem:
    def __init__(self, initial, goal):
        self.initial = initial # Estado inicial
        self.goal = goal # Meta
    
    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Función de desempeño
    def is_goal(self, state):
        return self.goal == state
    
    def action_cost(self, state1, action, state2):
        return 1
   
    def h(self, state):
        return 0



In [3]:
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph
    
    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

In [54]:
metro_cdmx = {
    # === LÍNEA 1 
    'Observatorio': {'Tacubaya': 1.15},
    'Tacubaya': {'Observatorio': 1.15, 'Juanacatlán': 1.15, 'Constituyentes': 1.00, 'San Pedro de los Pinos': 1.10, 'Chapultepec': 0.90, 'Patriotismo': 1.10},
    'Juanacatlán': {'Tacubaya': 1.15, 'Chapultepec': 0.70},
    'Chapultepec': {'Juanacatlán': 0.70, 'Sevilla': 0.50, 'Tacubaya': 0.90},
    'Sevilla': {'Chapultepec': 0.50, 'Insurgentes': 0.60},
    'Insurgentes': {'Sevilla': 0.60, 'Cuauhtémoc': 0.75},
    'Cuauhtémoc': {'Insurgentes': 0.75, 'Balderas': 0.45},
    'Balderas': {'Cuauhtémoc': 0.45, 'Salto del Agua': 0.45, 'Juárez': 0.65, 'Niños Héroes': 0.70},
    'Salto del Agua': {'Balderas': 0.45, 'Isabel la Católica': 0.45, 'San Juan de Letrán': 0.40, 'Doctores': 0.55},
    'Isabel la Católica': {'Salto del Agua': 0.45, 'Pino Suárez': 0.40},
    'Pino Suárez': {'Isabel la Católica': 0.40, 'Merced': 0.75, 'Zócalo/Tenochtitlan': 0.75, 'San Antonio Abad': 0.80},
    'Merced': {'Pino Suárez': 0.75, 'Candelaria': 0.70},
    'Candelaria': {'Merced': 0.70, 'San Lázaro': 0.85, 'Morelos': 1.10, 'Fray Servando': 0.65},
    'San Lázaro': {'Candelaria': 0.85, 'Moctezuma': 0.50, 'Morelos': 1.20, 'Ricardo Flores Magón': 1.05},
    'Moctezuma': {'San Lázaro': 0.50, 'Balbuena': 0.70},
    'Balbuena': {'Moctezuma': 0.70, 'Boulevard Puerto Aéreo': 0.60},
    'Boulevard Puerto Aéreo': {'Balbuena': 0.60, 'Gómez Farías': 0.60},
    'Gómez Farías': {'Boulevard Puerto Aéreo': 0.60, 'Zaragoza': 0.70},
    'Zaragoza': {'Gómez Farías': 0.70, 'Pantitlán': 1.30},
    'Pantitlán': {'Zaragoza': 1.30, 'Puebla': 1.40, 'Hangares': 1.60, 'Agrícola Oriental': 1.40},

    # === LÍNEA 2 
    'Cuatro Caminos': {'Panteones': 1.65},
    'Panteones': {'Cuatro Caminos': 1.65, 'Tacuba': 1.35},
    'Tacuba': {'Panteones': 1.35, 'Cuitláhuac': 0.65, 'Refinería': 1.25, 'San Joaquín': 1.15, 'Popotla': 0.60},
    'Cuitláhuac': {'Tacuba': 0.65, 'Popotla': 0.60},
    'Popotla': {'Cuitláhuac': 0.60, 'Colegio Militar': 0.45, 'Tacuba': 0.60},
    'Colegio Militar': {'Popotla': 0.45, 'Normal': 0.55},
    'Normal': {'Colegio Militar': 0.55, 'San Cosme': 0.65},
    'San Cosme': {'Normal': 0.65, 'Revolución': 0.70},
    'Revolución': {'San Cosme': 0.70, 'Hidalgo': 0.55},
    'Hidalgo': {'Revolución': 0.55, 'Bellas Artes': 0.45, 'Guerrero': 0.70, 'Juárez': 0.50},
    'Bellas Artes': {'Hidalgo': 0.45, 'Allende': 0.40, 'Garibaldi/Lagunilla': 0.60, 'San Juan de Letrán': 0.45},
    'Allende': {'Bellas Artes': 0.40, 'Zócalo/Tenochtitlan': 0.60},
    'Zócalo/Tenochtitlan': {'Allende': 0.60, 'Pino Suárez': 0.75},
    'San Antonio Abad': {'Pino Suárez': 0.80, 'Chabacano': 0.65},
    'Chabacano': {'San Antonio Abad': 0.65, 'Viaducto': 0.75, 'Lázaro Cárdenas': 1.00, 'Jamaica': 1.00, 'La Viga': 0.80, 'Eje Central': 0.90},
    'Viaducto': {'Chabacano': 0.75, 'Xola': 0.80},
    'Xola': {'Viaducto': 0.80, 'Villa de Cortés': 0.70},
    'Villa de Cortés': {'Xola': 0.70, 'Nativitas': 0.75},
    'Nativitas': {'Villa de Cortés': 0.75, 'Portales': 0.90},
    'Portales': {'Nativitas': 0.90, 'Ermita': 0.75},
    'Ermita': {'Portales': 0.75, 'General Anaya': 0.95, 'Eje Central': 0.85, 'Mexicaltzingo': 1.80},
    'General Anaya': {'Ermita': 0.95, 'Tasqueña': 1.30},
    'Tasqueña': {'General Anaya': 1.30},

    # === LÍNEA 3 
    'Indios Verdes': {'Deportivo 18 de Marzo': 1.15},
    'Deportivo 18 de Marzo': {'Indios Verdes': 1.15, 'Potrero': 0.95, 'La Villa/Basílica': 0.85, 'Martín Carrera': 1.40, 'Lindavista': 1.00},
    'Potrero': {'Deportivo 18 de Marzo': 0.95, 'La Raza': 1.10},
    'La Raza': {'Potrero': 1.10, 'Tlatelolco': 1.30, 'Autobuses del Norte': 0.95, 'Misterios': 1.10, 'Vallejo': 1.20},
    'Tlatelolco': {'La Raza': 1.30, 'Guerrero': 0.80},
    'Guerrero': {'Tlatelolco': 0.80, 'Hidalgo': 0.70, 'Buenavista': 0.75, 'Garibaldi/Lagunilla': 0.75},
    'Juárez': {'Hidalgo': 0.50, 'Balderas': 0.65},
    'Niños Héroes': {'Balderas': 0.70, 'Hospital General': 0.60},
    'Hospital General': {'Niños Héroes': 0.60, 'Centro Médico': 0.65},
    'Centro Médico': {'Hospital General': 0.65, 'Etiopía/Plaza de la Transparencia': 0.80, 'Lázaro Cárdenas': 0.70, 'Chilpancingo': 1.15},
    'Etiopía/Plaza de la Transparencia': {'Centro Médico': 0.80, 'Eugenia': 0.70},
    'Eugenia': {'Etiopía/Plaza de la Transparencia': 0.70, 'División del Norte': 0.70},
    'División del Norte': {'Eugenia': 0.70, 'Zapata': 0.80},
    'Zapata': {'División del Norte': 0.80, 'Coyoacán': 1.15, 'Parque de los Venados': 0.75, 'Hospital 20 de Noviembre': 0.80},
    'Coyoacán': {'Zapata': 1.15, 'Viveros/Derechos Humanos': 0.90},
    'Viveros/Derechos Humanos': {'Coyoacán': 0.90, 'Miguel Ángel de Quevedo': 0.85},
    'Miguel Ángel de Quevedo': {'Viveros/Derechos Humanos': 0.85, 'Copilco': 1.25},
    'Copilco': {'Miguel Ángel de Quevedo': 1.25, 'Universidad': 1.30},
    'Universidad': {'Copilco': 1.30},

    # === LÍNEA 4 
    'Martín Carrera': {'Talismán': 1.10, 'Deportivo 18 de Marzo': 1.40, 'La Villa/Basílica': 1.00},
    'Talismán': {'Martín Carrera': 1.10, 'Consulado': 0.95},
    'Consulado': {'Talismán': 0.95, 'Canal del Norte': 0.90, 'Bondojito': 0.80, 'Eduardo Molina': 0.85},
    'Canal del Norte': {'Consulado': 0.90, 'Morelos': 0.80},
    'Morelos': {'Canal del Norte': 0.80, 'Candelaria': 1.05, 'San Lázaro': 1.20, 'Tepito': 0.85},
    'Fray Servando': {'Candelaria': 0.65, 'Jamaica': 0.90},
    'Jamaica': {'Fray Servando': 0.90, 'Santa Anita': 0.75, 'Chabacano': 1.00, 'Mixiuhca': 0.85},
    'Santa Anita': {'Jamaica': 0.75, 'La Viga': 0.85},

    # === LÍNEA 5 
    'Politécnico': {'Instituto del Petróleo': 1.10},
    'Instituto del Petróleo': {'Politécnico': 1.10, 'Autobuses del Norte': 1.05, 'Lindavista': 1.15, 'Vallejo': 0.90},
    'Autobuses del Norte': {'Instituto del Petróleo': 1.05, 'La Raza': 0.95},
    'Misterios': {'La Raza': 1.10, 'Consulado': 1.50},
    'Vallejo': {'Instituto del Petróleo': 0.90, 'La Raza': 1.20},
    'Bondojito': {'Consulado': 0.80, 'Aragón': 0.95},
    'Aragón': {'Bondojito': 0.95, 'Eduardo Molina': 0.80},
    'Eduardo Molina': {'Aragón': 0.80, 'Consulado': 0.85, 'Terminal Aérea': 1.30},
    'Oceanía': {'Deportivo Oceanía': 0.85, 'Terminal Aérea': 1.15, 'Bosque de Aragón': 1.10},
    'Terminal Aérea': {'Oceanía': 1.15, 'Hangares': 0.95},
    'Hangares': {'Terminal Aérea': 0.95, 'Pantitlán': 1.60},

    # === LÍNEA 6 
    'El Rosario': {'Tezozómoc': 1.10, 'Aquiles Serdán': 1.60},
    'Tezozómoc': {'El Rosario': 1.10, 'Azcapotzalco': 1.00},
    'Azcapotzalco': {'Tezozómoc': 1.00, 'Ferrería/Arena CDMX': 1.20},
    'Ferrería/Arena CDMX': {'Azcapotzalco': 1.20, 'Norte 45': 1.10},
    'Norte 45': {'Ferrería/Arena CDMX': 1.10, 'Instituto del Petróleo': 1.20},
    'Lindavista': {'Instituto del Petróleo': 1.15, 'Deportivo 18 de Marzo': 1.00},
    'La Villa/Basílica': {'Deportivo 18 de Marzo': 0.85, 'Martín Carrera': 1.00},

    # === LÍNEA 7 
    'Aquiles Serdán': {'El Rosario': 1.60, 'Camarones': 1.30},
    'Camarones': {'Aquiles Serdán': 1.30, 'Refinería': 1.10},
    'Refinería': {'Camarones': 1.10, 'Tacuba': 1.25},
    'San Joaquín': {'Tacuba': 1.15, 'Polanco': 1.20},
    'Polanco': {'San Joaquín': 1.20, 'Auditorio': 0.90},
    'Auditorio': {'Polanco': 0.90, 'Constituyentes': 1.30},
    'Constituyentes': {'Auditorio': 1.30, 'Tacubaya': 1.00},
    'San Pedro de los Pinos': {'Tacubaya': 1.10, 'San Antonio': 1.00},
    'San Antonio': {'San Pedro de los Pinos': 1.00, 'Mixcoac': 1.25},
    'Mixcoac': {'San Antonio': 1.25, 'Barranca del Muerto': 1.50, 'Insurgentes Sur': 1.10},
    'Barranca del Muerto': {'Mixcoac': 1.50},

    # === LÍNEA 8 
    'Garibaldi/Lagunilla': {'Bellas Artes': 0.60, 'Guerrero': 0.75, 'Lagunilla': 0.50, 'Tepito': 0.70},
    'San Juan de Letrán': {'Bellas Artes': 0.45, 'Salto del Agua': 0.40},
    'Doctores': {'Salto del Agua': 0.55, 'Obrera': 0.70},
    'Obrera': {'Doctores': 0.70, 'Chabacano': 0.80},
    'La Viga': {'Chabacano': 0.80, 'Santa Anita': 0.85},
    'Plutarco Elías Calles': {'Santa Anita': 1.10, 'Coyuya': 0.90},
    'Coyuya': {'Plutarco Elías Calles': 0.90, 'Iztacalco': 1.00},
    'Iztacalco': {'Coyuya': 1.00, 'Apatlaco': 0.90},
    'Apatlaco': {'Iztacalco': 0.90, 'Aculco': 1.00},
    'Aculco': {'Apatlaco': 1.00, 'Escuadrón 201': 0.90},
    'Escuadrón 201': {'Aculco': 0.90, 'Atlalilco': 1.70},
    'Atlalilco': {'Escuadrón 201': 1.70, 'Iztapalapa': 1.50, 'Culhuacán': 1.80, 'Mexicaltzingo': 1.60},
    'Iztapalapa': {'Atlalilco': 1.50, 'Cerro de la Estrella': 0.90},
    'Cerro de la Estrella': {'Iztapalapa': 0.90, 'UAM-I': 1.20},
    'UAM-I': {'Cerro de la Estrella': 1.20, 'Constitución de 1917': 1.40},
    'Constitución de 1917': {'UAM-I': 1.40},

    # === LÍNEA 9 
    'Patriotismo': {'Tacubaya': 1.10, 'Chilpancingo': 1.00},
    'Chilpancingo': {'Patriotismo': 1.00, 'Centro Médico': 1.15},
    'Lázaro Cárdenas': {'Centro Médico': 0.70, 'Chabacano': 1.00},
    'Mixiuhca': {'Jamaica': 0.85, 'Velódromo': 1.00},
    'Velódromo': {'Mixiuhca': 1.00, 'Ciudad Deportiva': 1.10},
    'Ciudad Deportiva': {'Velódromo': 1.10, 'Puebla': 0.90},
    'Puebla': {'Ciudad Deportiva': 0.90, 'Pantitlán': 1.40},

    # === LÍNEA 10 
    'Agrícola Oriental': {'Pantitlán': 1.40, 'Canal de San Juan': 1.10},
    'Canal de San Juan': {'Agrícola Oriental': 1.10, 'Tepalcates': 1.30},
    'Tepalcates': {'Canal de San Juan': 1.30, 'Guelatao': 1.20},
    'Guelatao': {'Tepalcates': 1.20, 'Peñón Viejo': 2.00},
    'Peñón Viejo': {'Guelatao': 2.00, 'Acatitla': 1.40},
    'Acatitla': {'Peñón Viejo': 1.40, 'Santa Marta': 1.20},
    'Santa Marta': {'Acatitla': 1.20, 'Los Reyes': 1.80},
    'Los Reyes': {'Santa Marta': 1.80, 'La Paz': 1.90},
    'La Paz': {'Los Reyes': 1.90},

    # === LÍNEA 11 
    'Buenavista': {'Guerrero': 0.75},
    'Lagunilla': {'Garibaldi/Lagunilla': 0.50, 'Tepito': 0.60},
    'Tepito': {'Lagunilla': 0.60, 'Morelos': 0.85, 'Garibaldi/Lagunilla': 0.70},
    'Ricardo Flores Magón': {'San Lázaro': 1.05, 'Oceanía': 1.60},
    'Deportivo Oceanía': {'Oceanía': 0.85, 'Bosque de Aragón': 1.10},
    'Bosque de Aragón': {'Deportivo Oceanía': 1.10, 'Villa de Aragón': 0.80,'Oceanía': 1.10},
    'Villa de Aragón': {'Bosque de Aragón': 0.80, 'Nezahualcóyotl': 1.30},
    'Nezahualcóyotl': {'Villa de Aragón': 1.30, 'Impulsora': 1.40},
    'Impulsora': {'Nezahualcóyotl': 1.40, 'Río de los Remedios': 0.90},
    'Río de los Remedios': {'Impulsora': 0.90, 'Múzquiz': 1.10},
    'Múzquiz': {'Río de los Remedios': 1.10, 'Ecatepec': 1.50},
    'Ecatepec': {'Múzquiz': 1.50, 'Olímpica': 0.80},
    'Olímpica': {'Ecatepec': 0.80, 'Plaza Aragón': 0.80},
    'Plaza Aragón': {'Olímpica': 0.80, 'Ciudad Azteca': 0.70},
    'Ciudad Azteca': {'Plaza Aragón': 0.70},
    
    # === LINEA 12 
    'Insurgentes Sur': {'Mixcoac': 1.10, 'Hospital 20 de Noviembre': 0.90},
    'Hospital 20 de Noviembre': {'Insurgentes Sur': 0.90, 'Zapata': 0.80},
    'Parque de los Venados': {'Zapata': 0.75, 'Eje Central': 1.10},
    'Eje Central': {'Parque de los Venados': 1.10, 
    'Ermita': 0.85, 'Chabacano': 0.90},
    'Mexicaltzingo': {'Ermita': 1.80, 'Atlalilco': 1.60},
    'Culhuacán': {'Atlalilco': 1.80, 'San Andrés Tomatlán': 0.90},
    'San Andrés Tomatlán': {'Culhuacán': 0.90, 'Lomas de Estrella': 1.10},
    'Lomas de Estrella': {'San Andrés Tomatlán': 1.10, 'Calle 11': 0.90},
    'Calle 11': {'Lomas de Estrella': 0.90, 'Periférico Oriente': 1.30},
    'Periférico Oriente': {'Calle 11': 1.30, 'Tezonco': 1.40},
    'Tezonco': {'Periférico Oriente': 1.40, 'Olivos': 0.80},
    'Olivos': {'Tezonco': 0.80, 'Nopalera': 1.30},
    'Nopalera': {'Olivos': 1.30, 'Zapotitlán': 1.20},
    'Zapotitlán': {'Nopalera': 1.20, 'Tlaltenco': 1.30},
    'Tlaltenco': {'Zapotitlán': 1.30, 'Tláhuac': 1.30},
    'Tláhuac': {'Tlaltenco': 1.30}}


In [44]:
class Node:
    def __init__(self, state, parent = None, action = None, path_cost = 0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)


In [60]:
def depth_first_graph_search(problem):
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node
    frontier = [start_node]
    explored = set()

    while frontier:
        node = frontier.pop()
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)
    return None

In [49]:
from collections import deque

def breadth_first_graph_search(problem):
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node
    frontier = deque([start_node])
    explored = set()

    while frontier:
        node = frontier.popleft()
        explored.add(node.state)

        if problem.is_goal(node.state):
            return node

        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)
    return None

In [61]:
problem = GraphProblem("Cuatro Caminos" , "Pantitlán", metro_cdmx)
nodo_solucion = depth_first_graph_search(problem)

if nodo_solucion:
    ruta = nodo_solucion.path()
    costo_total = nodo_solucion.path_cost
    
    print(f"Ruta encontrada: {ruta}")
    print(f"Costo total acumulado: {costo_total}")
else:
    print("Error denttro del Algoritmo")


Ruta encontrada: ['Cuatro Caminos', 'Panteones', 'Tacuba', 'Popotla', 'Colegio Militar', 'Normal', 'San Cosme', 'Revolución', 'Hidalgo', 'Juárez', 'Balderas', 'Niños Héroes', 'Hospital General', 'Centro Médico', 'Chilpancingo', 'Patriotismo', 'Tacubaya', 'San Pedro de los Pinos', 'San Antonio', 'Mixcoac', 'Insurgentes Sur', 'Hospital 20 de Noviembre', 'Zapata', 'Parque de los Venados', 'Eje Central', 'Chabacano', 'La Viga', 'Santa Anita', 'Jamaica', 'Mixiuhca', 'Velódromo', 'Ciudad Deportiva', 'Puebla', 'Pantitlán']
Costo total acumulado: 29.400000000000002


In [65]:
nodo_solucion = breadth_first_graph_search(problem)

if nodo_solucion:
    ruta = nodo_solucion.path()
    costo_total = nodo_solucion.path_cost
    
    print(f"Ruta encontrada: {ruta}")
    print(f"Costo total acumulado: {costo_total}")
else:
    print("Error denttro del Algoritmo")


Ruta encontrada: ['Cuatro Caminos', 'Panteones', 'Tacuba', 'Popotla', 'Colegio Militar', 'Normal', 'San Cosme', 'Revolución', 'Hidalgo', 'Guerrero', 'Tlatelolco', 'La Raza', 'Misterios', 'Consulado', 'Eduardo Molina', 'Terminal Aérea', 'Hangares', 'Pantitlán']
Costo total acumulado: 16.6


In [70]:
problem2 = GraphProblem("Politécnico" , "Taxqueña", metro_cdmx)
nodo_solucion2 = depth_first_graph_search(problem2)

if nodo_solucion2:
    ruta = nodo_solucion.path()
    costo_total = nodo_solucion.path_cost
    
    print(f"Ruta encontrada: {ruta}")
    print(f"Costo total acumulado: {costo_total}")
else:
    print("Error denttro del Algoritmo")


Error denttro del Algoritmo


In [69]:
problem2 = GraphProblem("Zapata" , "Oceania", metro_cdmx)
nodo_solucion2 = depth_first_graph_search(problem2)

if nodo_solucion2:
    ruta = nodo_solucion.path()
    costo_total = nodo_solucion.path_cost
    
    print(f"Ruta encontrada: {ruta}")
    print(f"Costo total acumulado: {costo_total}")
else:
    print("Error denttro del Algoritmo")


Error denttro del Algoritmo
